# Tutorial: BDS 0.2.3 Trust Diagnostics

This notebook exercises the local `battery-data-standard` working tree with test fixtures. It focuses on the trust-building features added after 0.2.2: Repower/PEC public fixtures, optional adjacent-point current-sign sanity metadata, step/cycle semantic warnings, and explain/audit visibility.

Audience:
- BDS maintainers and reviewers checking the 0.2.3 candidate behavior.

Prerequisites:
- Run from the repository root or keep the repository layout unchanged.
- The local `.venv` or notebook kernel should have the package dependencies installed.

Learning goals:
- Confirm Repower and PEC fixtures detect and validate.
- Inspect current-sign and step/cycle diagnostics in conversion metadata and audit records.
- See the exact warning codes that downstream pipelines can branch on.


## Outline

1. Set up local imports and paths.
2. Convert the new Repower and PEC public fixtures.
3. Audit the full `tests/fixtures` corpus.
4. Recreate the edge-case datasets used by tests for current sign and step/cycle warnings.
5. Inspect the same diagnostics through `bds.explain()`.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find repository root")


repo_root = find_repo_root(Path.cwd().resolve())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import bds
from battery_data_standard.audit import audit

fixture_root = repo_root / "tests" / "fixtures"
work_dir = repo_root / "tmp" / "notebook_trust_diagnostics"
work_dir.mkdir(parents=True, exist_ok=True)

print("Repository root:", repo_root)
print("BDS version:", bds.__version__)
print("Fixture root:", fixture_root)
print("Notebook scratch dir:", work_dir)


Repository root: C:\work\Batterydatastandard
BDS version: 0.2.2
Fixture root: C:\work\Batterydatastandard\tests\fixtures
Notebook scratch dir: C:\work\Batterydatastandard\tmp\notebook_trust_diagnostics


## 1. Public Fixture Coverage

The new Repower and PEC files live under `tests/fixtures/<cycler>`. Each directory has a small raw file plus a manifest that tells the fixture test what to expect.


In [2]:
for cycler in ("repower", "pec"):
    fixture_dir = fixture_root / cycler
    print(f"\n[{cycler}] files")
    for item in sorted(fixture_dir.iterdir()):
        print("-", item.relative_to(repo_root), item.stat().st_size, "bytes")
    manifest = fixture_dir / "manifest.jsonl"
    for line in manifest.read_text(encoding="utf-8").splitlines():
        print("manifest:", line)



[repower] files
- tests\fixtures\repower\basic.csv 238 bytes
- tests\fixtures\repower\manifest.jsonl 251 bytes
manifest: {"name":"repower_basic_csv","input":"basic.csv","cycler_arg":"auto","expected_cycler":"repower","current_sign":"charge-positive","repair_policy":"warn","min_rows":2,"required_columns":["test_time_s","voltage_v","current_a"],"expected_first_time":0.0}

[pec] files
- tests\fixtures\pec\basic.csv 261 bytes
- tests\fixtures\pec\manifest.jsonl 236 bytes
manifest: {"name":"pec_basic_csv","input":"basic.csv","cycler_arg":"auto","expected_cycler":"pec","current_sign":"preserve","repair_policy":"warn","min_rows":2,"required_columns":["test_time_s","voltage_v","current_a"],"expected_first_time":0.0}


## 2. Convert Repower and PEC Fixtures

This mirrors the parameterized fixture test: detect the source, read it with the requested current-sign option, print the normalized rows, and show the new trust diagnostics in metadata.


In [3]:
for cycler in ("repower", "pec"):
    manifest_path = fixture_root / cycler / "manifest.jsonl"
    case = json.loads(manifest_path.read_text(encoding="utf-8").splitlines()[0])
    source = fixture_root / cycler / case["input"]

    detection = bds.detect(source)
    df, report = bds.read_with_report(
        source,
        cycler=case.get("cycler_arg", "auto"),
        current_sign=case.get("current_sign", "charge-positive"),
        repair_policy=case.get("repair_policy", "warn"),
        strict=False,
    )

    print(f"\n=== {cycler.upper()} ===")
    print("source:", source.relative_to(repo_root))
    print("detected:", detection.cycler, "confidence:", detection.confidence)
    print("report cycler:", report.cycler)
    print("rows:", df.height)
    print("columns:", df.columns)
    print("current_sign:", report.current_sign)
    print("current_sign_confidence:", report.metadata.get("current_sign_confidence"))
    print("current_sign_sanity:", json.dumps(report.metadata.get("current_sign_sanity"), indent=2))
    print("semantic_sources:", json.dumps(report.metadata.get("semantic_sources"), indent=2))
    print("normalized preview:")
    print(df.head(5))



=== REPOWER ===
source: tests\fixtures\repower\basic.csv
detected: repower confidence: 0.8
report cycler: repower
rows: 2
columns: ['test_time_s', 'voltage_v', 'current_a', 'unix_time_s', 'date_time', 'cycle_index', 'step_index', 'ambient_temperature_deg_c', 'charge_capacity_ah', 'discharge_capacity_ah', 'power_w']
current_sign: charge-positive
current_sign_confidence: disabled
current_sign_sanity: {
  "status": "disabled",
  "method": "none",
  "confidence": "disabled",
  "reason": "current sign sanity check was disabled",
  "evidence": []
}
semantic_sources: {
  "test_time_s": {
    "origin": "source",
    "source": "Relative Time(Sec)",
    "transform": "numeric"
  },
  "cycle_index": {
    "origin": "source",
    "source": "Cycle ID",
    "transform": "integer"
  },
  "step_index": {
    "origin": "source",
    "source": "Step ID",
    "transform": "integer"
  }
}
normalized preview:
shape: (2, 11)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬─────

## 3. Audit the Test Fixture Corpus

The audit report includes step/cycle diagnostics by default. Current-sign sanity is disabled by default and becomes visible when `current_sign_check="adjacent"` is requested.


In [4]:
audit_report = audit(fixture_root, recursive=True)
print("files:", audit_report.files)
print("converted:", audit_report.converted)
print("unsupported:", audit_report.unsupported)
print("errors:", audit_report.errors)
print("top_issue_codes:", audit_report.top_issue_codes)

for target in ("repower/basic.csv", "pec/basic.csv"):
    record = next(r for r in audit_report.records if r.relative_path.replace("\\", "/") == target)
    print(f"\n=== audit record: {target} ===")
    print("status:", record.status)
    print("quality:", record.quality_score, record.quality_grade)
    print("issue codes:", [issue.code for issue in record.issues])
    print("current_sign_confidence:", record.checks.get("current_sign_confidence"))
    print("current_sign_sanity:", json.dumps(record.checks.get("current_sign_sanity"), indent=2))
    print("step_cycle_semantics:", json.dumps(record.checks.get("step_cycle_semantics"), indent=2))


files: 9
converted: 9
unsupported: 0
errors: 0
top_issue_codes: {'inferred-step-cycle-semantics': 1, 'step-transition-discontinuity': 1}

=== audit record: repower/basic.csv ===
status: converted
quality: 100 A
issue codes: []
current_sign_confidence: disabled
current_sign_sanity: {
  "status": "disabled",
  "method": "none",
  "confidence": "disabled",
  "reason": "current sign sanity check was disabled",
  "evidence": []
}
step_cycle_semantics: {
  "repeated_step_segments": 0,
  "step_transition_discontinuities": 0,
  "examples": [],
  "inferred_fields": [],
  "semantic_sources": {
    "test_time_s": {
      "origin": "source",
      "source": "Relative Time(Sec)",
      "transform": "numeric"
    },
    "cycle_index": {
      "origin": "source",
      "source": "Cycle ID",
      "transform": "integer"
    },
    "step_index": {
      "origin": "source",
      "source": "Step ID",
      "transform": "integer"
    }
  },
  "repeated_step_examples": [],
  "step_transition_examples": []

## 4. Edge-Case Diagnostics from the Test Suite

These tiny CSV files recreate the same scenarios covered by `tests/test_supporting_modules.py`: an opt-in adjacent-point current-sign conflict, a repeated step segment, a step transition discontinuity, and inferred whole-test time.


In [5]:
edge_cases = {
    "sign_conflict.csv": """Test Time (s),Voltage (V),Current (A),Discharging Capacity (Ah)\n0,3.40,-1.0,0.00\n1,3.45,-1.0,0.01\n2,3.50,-1.0,0.02\n3,3.55,-1.0,0.03\n""",
    "repeated_steps.csv": """Test Time (s),Voltage (V),Current (A),Cycle Count,Step Index\n0,3.40,0.1,1,1\n1,3.41,0.1,1,1\n2,3.42,0.1,1,2\n3,3.43,0.1,1,2\n4,3.44,0.1,1,1\n5,3.45,0.1,1,1\n""",
    "step_transition.csv": """Test Time (s),Step Time (s),Voltage (V),Current (A),Cycle Count,Step Index\n0,0,3.40,0.1,1,1\n1,1,3.41,0.1,1,1\n2,2,3.42,0.1,1,2\n3,3,3.43,0.1,1,2\n""",
    "inferred_time.csv": """Step Time (s),Voltage (V),Current (A),Step Index\n0,3.40,0.1,1\n1,3.41,0.1,1\n0,3.42,0.1,2\n1,3.43,0.1,2\n""",
}

for filename, text in edge_cases.items():
    (work_dir / filename).write_text(text, encoding="utf-8")

print("Wrote edge-case files:")
for path in sorted(work_dir.glob("*.csv")):
    print("-", path.relative_to(repo_root))


Wrote edge-case files:
- tmp\notebook_trust_diagnostics\inferred_time.csv
- tmp\notebook_trust_diagnostics\repeated_steps.csv
- tmp\notebook_trust_diagnostics\sign_conflict.csv
- tmp\notebook_trust_diagnostics\step_transition.csv


In [6]:
scenarios = [
    ("sign_conflict.csv", "charge-positive", "current-sign-suspicious"),
    ("repeated_steps.csv", "preserve", "repeated-step-segments"),
    ("step_transition.csv", "preserve", "step-transition-discontinuity"),
    ("inferred_time.csv", "preserve", "inferred-step-cycle-semantics"),
]

for filename, current_sign, expected_code in scenarios:
    source = work_dir / filename
    report = audit(source, cycler="generic", current_sign=current_sign, current_sign_check="adjacent")
    record = report.records[0]
    codes = [issue.code for issue in record.issues]
    print(f"\n=== {filename} ===")
    print("expected code:", expected_code)
    print("observed codes:", codes)
    print("status:", record.status)
    print("quality:", record.quality_score, record.quality_grade)
    print("current_sign_confidence:", record.checks.get("current_sign_confidence"))
    print("current_sign_sanity:", json.dumps(record.checks.get("current_sign_sanity"), indent=2))
    print("step_cycle_semantics:", json.dumps(record.checks.get("step_cycle_semantics"), indent=2))
    assert expected_code in codes



=== sign_conflict.csv ===
expected code: current-sign-suspicious
observed codes: ['current-sign-suspicious']
status: converted
quality: 82 B
current_sign_confidence: high
current_sign_sanity: {
  "status": "suspicious",
  "method": "adjacent",
  "confidence": "high",
  "reason": "row 0->1: discharge current with voltage increase 0.05 V; row 1->2: discharge current with voltage increase 0.05 V; row 2->3: discharge current with voltage increase 0.05 V",
  "evidence": [
    "row 0->1: discharge current with voltage increase 0.05 V",
    "row 1->2: discharge current with voltage increase 0.05 V",
    "row 2->3: discharge current with voltage increase 0.05 V"
  ],
  "agreements": [],
  "checked_pairs": 3,
  "conflict_pairs": 3,
  "agreement_pairs": 0,
  "conflict_fraction": 1.0
}
step_cycle_semantics: {
  "repeated_step_segments": 0,
  "step_transition_discontinuities": 0,
  "examples": [],
  "inferred_fields": [],
  "semantic_sources": {
    "test_time_s": {
      "origin": "source",
    

## 5. Conversion Metadata and Explain Output

`read_with_report()` and `bds.explain()` expose the same diagnostic payloads so automated pipelines and humans see the same evidence.


In [7]:
source = work_dir / "sign_conflict.csv"
df, conversion_report = bds.read_with_report(
    source,
    cycler="generic",
    current_sign="charge-positive",
    current_sign_check="adjacent",
    strict=False,
)
print("conversion metadata excerpt:")
for key in ("current_sign_confidence", "current_sign_sanity", "semantic_sources", "step_cycle_semantics"):
    print(f"\n{key}:")
    print(json.dumps(conversion_report.metadata.get(key), indent=2))

explain_report = bds.explain(source, cycler="generic", current_sign="charge-positive", current_sign_check="adjacent")
print("\nexplain text:")
print(explain_report.to_text())


conversion metadata excerpt:

current_sign_confidence:
"high"

current_sign_sanity:
{
  "status": "suspicious",
  "method": "adjacent",
  "confidence": "high",
  "reason": "row 0->1: discharge current with voltage increase 0.05 V; row 1->2: discharge current with voltage increase 0.05 V; row 2->3: discharge current with voltage increase 0.05 V",
  "evidence": [
    "row 0->1: discharge current with voltage increase 0.05 V",
    "row 1->2: discharge current with voltage increase 0.05 V",
    "row 2->3: discharge current with voltage increase 0.05 V"
  ],
  "agreements": [],
  "checked_pairs": 3,
  "conflict_pairs": 3,
  "agreement_pairs": 0,
  "conflict_fraction": 1.0
}

semantic_sources:
{
  "test_time_s": {
    "origin": "source",
    "source": "Test Time (s)",
    "transform": "numeric"
  },
  "cycle_index": {
    "origin": "absent",
    "source": null,
    "transform": null
  },
  "step_index": {
    "origin": "absent",
    "source": null,
    "transform": null
  }
}

step_cycle_sem

## Exercise

Try changing `current_sign_check="adjacent"` in the previous cell to `"none"`. The current-sign diagnostic should become disabled, which is the default for large or heuristic-free workflows.

Common pitfall: these checks are conservative warnings. They do not repair current sign or step semantics automatically; they tell you where to review the raw file, procedure, and conversion report before downstream aggregation.
